# SEC RAG Pipeline — CS 6120 NLP

**Teammate 1 deliverables**: Download → Clean → Chunk → Embed → Load → Demo queries

**Corpus**: 30 companies × 5 sectors × 3 years (2021–2023), 10-K key sections + 10-Q  
**Target**: ~13,000–16,000 chunks  
**Runtime order**: Run cells top to bottom. Each step saves intermediate results to Google Drive so you can resume if the session drops.

---
### Steps
1. Install & Mount Drive
2. Download filings from SEC EDGAR
3. Clean HTML + extract key sections
4. Chunk (512 tokens, 64 overlap)
5. Embed with GPU (`all-MiniLM-L6-v2`)
6. Set up PostgreSQL + pgvector
7. Load data + build indexes
8. Run 10 demo queries (hybrid retrieval)

## Step 0 — Install dependencies & mount Drive
> Runtime → Change runtime type → **T4 GPU** before running

In [ ]:
!pip install -q \
    sec-edgar-downloader==0.9.0 \
    sentence-transformers \
    transformers \
    psycopg2-binary \
    beautifulsoup4 \
    lxml \
    tqdm \
    pgvector
print('Done.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/rag_project'
for d in ['data/raw', 'data/chunks', 'logs']:
    os.makedirs(f'{BASE_DIR}/{d}', exist_ok=True)
print(f'Base dir: {BASE_DIR}')

## Configuration — edit this cell only

In [ ]:
# ── Corpus ───────────────────────────────────────────────────────────────────
COMPANIES = {
    'banking':    ['JPM', 'BAC', 'WFC', 'C',    'GS',   'MS'],
    'technology': ['AAPL','MSFT','GOOGL','META','NVDA',  'AMD'],
    'healthcare': ['JNJ', 'UNH', 'PFE', 'ABBV','MRK',   'LLY'],
    'energy':     ['XOM', 'CVX', 'COP', 'SLB', 'EOG',   'PXD'],
    'consumer':   ['AMZN','WMT', 'HD',  'MCD', 'NKE',   'COST'],
}
DATE_START    = '2021-01-01'
DATE_END      = '2024-01-01'
FILING_TYPES  = ['10-K', '10-Q']
CHUNK_TOKENS  = 512
CHUNK_OVERLAP = 64
ALPHA         = 0.7   # hybrid weight: alpha*dense + (1-alpha)*BM25

# ── Derived ──────────────────────────────────────────────────────────────────
TICKER_TO_SECTOR = {t: s for s, tickers in COMPANIES.items() for t in tickers}
ALL_TICKERS      = list(TICKER_TO_SECTOR.keys())

# ── File paths ────────────────────────────────────────────────────────────────
CHUNKS_PATH = f'{BASE_DIR}/data/chunks/chunks.jsonl'
EMB_PATH    = f'{BASE_DIR}/embeddings.npy'

# ── DB (filled in Step 6) ─────────────────────────────────────────────────────
DB_CONFIG = {
    'host':     'localhost',
    'port':     5432,
    'dbname':   'rag_db',
    'user':     'rag_user',
    'password': 'rag_pass',
}

print(f'Companies : {len(ALL_TICKERS)}')
print(f'Date range: {DATE_START} → {DATE_END}')
print(f'Filing    : {FILING_TYPES}')

## Step 1 — Download SEC filings
Uses `sec-edgar-downloader`. Respects EDGAR rate limit (10 req/s).  
Files saved to `data/raw/sec-edgar-filings/{TICKER}/{FORM}/{accession}/`.  
**Expected time**: 30–60 min (EDGAR rate-limits to ~600 req/min).

In [ ]:
import time
from sec_edgar_downloader import Downloader

# !! Change to your name and email (required by EDGAR)
dl = Downloader('NEU_CS6120_Team', 'your-email@northeastern.edu',
                f'{BASE_DIR}/data/raw')

log_path = f'{BASE_DIR}/logs/download_log.txt'
failed   = []

with open(log_path, 'w') as log:
    for sector, tickers in COMPANIES.items():
        for ticker in tickers:
            for form in FILING_TYPES:
                try:
                    n = dl.get(form, ticker,
                               after=DATE_START, before=DATE_END)
                    msg = f'OK  {ticker:6s} {form:5s} {n} filing(s)'
                except Exception as e:
                    msg = f'ERR {ticker:6s} {form:5s} {e}'
                    failed.append((ticker, form, str(e)))
                print(msg)
                log.write(msg + '\n')
                time.sleep(0.5)   # 2 req/s — well under EDGAR limit

print(f'\nFailed downloads: {len(failed)}')
for f in failed:
    print(' ', f)

## Step 2 — Clean HTML + extract key sections

**10-K**: keep Item 1A (Risk Factors) + Item 7 (MD&A) + Item 8 (Financial Statements)  
**10-Q**: keep Item 1 (Financial Statements) + Item 2 (MD&A)

In [ ]:
import re
from bs4 import BeautifulSoup

# ── HTML cleaning ─────────────────────────────────────────────────────────────
def clean_html(html: str) -> str:
    soup = BeautifulSoup(html, 'lxml')
    # Remove non-content tags (XBRL inline, scripts, styles)
    for tag in soup(['script', 'style', 'ix:nonnumeric',
                     'ix:nonfraction', 'ix:header', 'ix:hidden']):
        tag.decompose()
    text = soup.get_text(separator='\n')
    text = re.sub(r'[ \t]+', ' ', text)          # collapse horizontal whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)       # max 2 consecutive newlines
    text = text.encode('ascii', 'ignore').decode()  # drop non-ASCII
    return text.strip()


# ── Section anchors ────────────────────────────────────────────────────────────
# Pattern: (section_name, regex).  Last match used to skip Table of Contents.
K10_ANCHORS = [
    ('risk_factors', r'item\s+1a[\s.\-]+\s*risk\s+factor'),
    ('mda',          r'item\s+7[\s.\-]+\s*management.{0,60}discussion'),
    ('financials',   r'item\s+8[\s.\-]+\s*financial\s+statement'),
    ('__end__',      r'item\s+9[\s.\-]+'),
]
Q10_ANCHORS = [
    ('financials',   r'item\s+1[\s.\-]+\s*financial\s+statement'),
    ('mda',          r'item\s+2[\s.\-]+\s*management.{0,60}discussion'),
    ('__end__',      r'item\s+3[\s.\-]+'),
]

def _find_anchors(text_lower: str, anchors):
    """Return sorted list of (char_pos, section_name) for found anchors."""
    positions = []
    for name, pat in anchors:
        matches = list(re.finditer(pat, text_lower))
        if matches:
            # Skip first match (usually Table of Contents entry)
            m = matches[-1] if len(matches) > 1 else matches[0]
            positions.append((m.start(), name))
    return sorted(positions)

def extract_sections(text: str, filing_type: str) -> str:
    """Extract only key sections from a cleaned filing text."""
    anchors   = K10_ANCHORS if filing_type == '10-K' else Q10_ANCHORS
    positions = _find_anchors(text.lower(), anchors)

    if len(positions) < 2:
        return text   # fallback: return full text

    parts = []
    for i, (start, name) in enumerate(positions):
        if name == '__end__':
            break
        end = positions[i + 1][0] if i + 1 < len(positions) else start + 60_000
        section = text[start:end].strip()
        if len(section) > 300:          # skip tiny/empty sections
            parts.append(section)

    return '\n\n'.join(parts) if parts else text

print('Cleaning functions ready.')

## Step 3 — Chunk (512 tokens, 64-token overlap, sentence-aware)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    'sentence-transformers/all-MiniLM-L6-v2')

def chunk_text(text: str,
               max_tokens: int = CHUNK_TOKENS,
               overlap: int    = CHUNK_OVERLAP) -> list[str]:
    """Sentence-aware chunking with token overlap."""
    # Split on sentence endings and paragraph breaks
    sentences = re.split(r'(?<=[.!?])\s+|\n\n+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]

    chunks, cur_sents, cur_len = [], [], 0

    for sent in sentences:
        sent_len = len(tokenizer.encode(sent, add_special_tokens=False))

        if cur_len + sent_len > max_tokens and cur_sents:
            chunks.append(' '.join(cur_sents))
            # Build overlap tail (last ~overlap tokens)
            tail, tail_len = [], 0
            for s in reversed(cur_sents):
                tl = len(tokenizer.encode(s, add_special_tokens=False))
                if tail_len + tl > overlap:
                    break
                tail.insert(0, s)
                tail_len += tl
            cur_sents, cur_len = tail, tail_len

        cur_sents.append(sent)
        cur_len += sent_len

    if cur_sents:
        chunks.append(' '.join(cur_sents))

    return [c for c in chunks if len(c.strip()) > 100]

print('Chunking function ready.')

In [ ]:
import json
from pathlib import Path
from tqdm import tqdm

raw_dir = Path(f'{BASE_DIR}/data/raw/sec-edgar-filings')
stats   = {'filings': 0, 'chunks': 0, 'errors': 0}

with open(CHUNKS_PATH, 'w') as out_f:
    for ticker in tqdm(ALL_TICKERS, desc='Tickers'):
        sector = TICKER_TO_SECTOR[ticker]

        for filing_type in FILING_TYPES:
            filing_dir = raw_dir / ticker / filing_type
            if not filing_dir.exists():
                continue

            for acc_dir in sorted(filing_dir.iterdir()):
                if not acc_dir.is_dir():
                    continue

                # Pick largest .htm file as the main document (skip index files)
                htm_files = [
                    f for f in (list(acc_dir.glob('*.htm')) +
                                list(acc_dir.glob('*.html')))
                    if 'index' not in f.name.lower()
                ]
                if not htm_files:
                    continue
                main_file = max(htm_files, key=lambda f: f.stat().st_size)

                accession = acc_dir.name          # e.g. 0000320193-22-000108

                # Approximate fiscal year from accession number
                try:
                    fiscal_year = 2000 + int(accession.split('-')[1][:2])
                    # 10-K filed in early year N covers fiscal year N-1
                    if filing_type == '10-K':
                        fiscal_year -= 1
                except Exception:
                    fiscal_year = None

                # Construct SEC EDGAR filing index URL (best-effort)
                index_files = list(acc_dir.glob('*-index.htm'))
                if index_files:
                    # Read CIK from the index page content
                    try:
                        idx_content = index_files[0].read_text(
                            encoding='utf-8', errors='ignore')
                        m = re.search(
                            r'/Archives/edgar/data/(\d+)/', idx_content)
                        cik = m.group(1) if m else 'unknown'
                    except Exception:
                        cik = 'unknown'
                    acc_nodash = accession.replace('-', '')
                    source_url = (f'https://www.sec.gov/Archives/edgar/data/'
                                  f'{cik}/{acc_nodash}/{accession}-index.htm')
                else:
                    # Fallback: EDGAR full-text search link
                    source_url = (f'https://efts.sec.gov/LATEST/search-index'
                                  f'?q=%22{accession}%22')

                try:
                    html  = main_file.read_text(encoding='utf-8', errors='ignore')
                    text  = clean_html(html)
                    text  = extract_sections(text, filing_type)
                    chnks = chunk_text(text)

                    stats['filings'] += 1
                    stats['chunks']  += len(chnks)

                    for i, content in enumerate(chnks):
                        record = {
                            'ticker':      ticker,
                            'sector':      sector,
                            'filing_type': filing_type,
                            'fiscal_year': fiscal_year,
                            'accession':   accession,
                            'source_url':  source_url,
                            'chunk_index': i,
                            'content':     content,
                        }
                        out_f.write(json.dumps(record) + '\n')

                except Exception as e:
                    stats['errors'] += 1
                    print(f'  ERR {ticker}/{filing_type}/{acc_dir.name}: {e}')

print(f"\nFilings processed : {stats['filings']}")
print(f"Total chunks      : {stats['chunks']}")
print(f"Errors            : {stats['errors']}")
print(f"Saved to          : {CHUNKS_PATH}")

## Step 4 — Embed with GPU (`all-MiniLM-L6-v2`, 384-dim)
Embeddings saved to Drive as `embeddings.npy` — no need to re-run if session drops.

In [ ]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

print(f'GPU available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU           : {torch.cuda.get_device_name(0)}')

# Load chunks
with open(CHUNKS_PATH) as f:
    all_chunks = [json.loads(l) for l in f]
print(f'Chunks loaded : {len(all_chunks)}')

model  = SentenceTransformer('all-MiniLM-L6-v2')
texts  = [c['content'] for c in all_chunks]

embeddings = model.encode(
    texts,
    batch_size=512,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # unit vectors → cosine = dot product
)

print(f'Shape: {embeddings.shape}')  # (N, 384)
np.save(EMB_PATH, embeddings)
print(f'Saved → {EMB_PATH}')

## Step 5 — PostgreSQL + pgvector setup (local in Colab)

> **Alternative**: Use an external DB (Supabase free tier supports pgvector). 
> Fill in `DB_CONFIG` in the config cell and skip this cell.

**Note**: local Colab PostgreSQL is lost when the session ends.  
For a persistent DB, use Supabase or export a pg_dump to Drive.

In [ ]:
import subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0 and result.stderr:
        print('STDERR:', result.stderr[:300])
    return result

# Install PostgreSQL
run('sudo apt-get install -y postgresql postgresql-contrib')
run('sudo service postgresql start')

# Create user + DB
run("sudo -u postgres psql -c \"CREATE USER rag_user WITH PASSWORD 'rag_pass';\"")
run("sudo -u postgres psql -c \"CREATE DATABASE rag_db OWNER rag_user;\"")
run("sudo -u postgres psql -c \"GRANT ALL PRIVILEGES ON DATABASE rag_db TO rag_user;\"")

# Install pgvector extension
r = run('sudo apt-get install -y postgresql-14-pgvector')
if r.returncode != 0:
    # Compile from source if apt package not found
    print('Compiling pgvector from source...')
    run('sudo apt-get install -y postgresql-server-dev-all build-essential')
    run('git clone --branch v0.7.4 https://github.com/pgvector/pgvector.git /tmp/pgvector')
    run('make -C /tmp/pgvector')
    run('sudo make -C /tmp/pgvector install')

print('PostgreSQL ready.')

## Step 6 — Create schema

In [ ]:
import psycopg2

conn = psycopg2.connect(**DB_CONFIG)
conn.autocommit = True
cur  = conn.cursor()

cur.execute('CREATE EXTENSION IF NOT EXISTS vector;')

cur.execute('''
DROP TABLE IF EXISTS chunks;
CREATE TABLE chunks (
    id           SERIAL      PRIMARY KEY,
    ticker       VARCHAR(10) NOT NULL,
    sector       VARCHAR(50) NOT NULL,
    fiscal_year  SMALLINT,
    quarter      SMALLINT,            -- NULL for 10-K
    filing_type  VARCHAR(10) NOT NULL,
    accession    VARCHAR(40),
    source_url   TEXT,
    content      TEXT        NOT NULL,
    embedding    VECTOR(384),
    content_tsv  TSVECTOR,
    chunk_index  SMALLINT
);
''')

# Auto-populate tsvector on insert/update (proposal requirement)
cur.execute('''
CREATE OR REPLACE FUNCTION chunks_tsv_trigger() RETURNS trigger AS $$
BEGIN
    NEW.content_tsv := to_tsvector('english', NEW.content);
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

DROP TRIGGER IF EXISTS chunks_tsv_update ON chunks;
CREATE TRIGGER chunks_tsv_update
    BEFORE INSERT OR UPDATE ON chunks
    FOR EACH ROW EXECUTE FUNCTION chunks_tsv_trigger();
''')

conn.autocommit = False
print('Schema + trigger created.')

## Step 7 — Load data + build indexes
**Rule**: build IVFFlat index **after** all rows are inserted (index quality depends on data distribution).

In [ ]:
from psycopg2.extras import execute_values

# Reload from Drive (safe to re-run)
with open(CHUNKS_PATH) as f:
    all_chunks = [json.loads(l) for l in f]
embeddings = np.load(EMB_PATH)

assert len(all_chunks) == len(embeddings), \
    f'Mismatch: {len(all_chunks)} chunks vs {len(embeddings)} embeddings'

rows = [
    (
        c['ticker'],
        c['sector'],
        c.get('fiscal_year'),
        c.get('quarter'),
        c['filing_type'],
        c.get('accession'),
        c.get('source_url'),
        c['content'],
        emb.tolist(),
        c['chunk_index'],
    )
    for c, emb in zip(all_chunks, embeddings)
]

BATCH = 500
for i in range(0, len(rows), BATCH):
    execute_values(
        cur,
        '''
        INSERT INTO chunks
            (ticker, sector, fiscal_year, quarter, filing_type,
             accession, source_url, content, embedding, chunk_index)
        VALUES %s
        ''',
        rows[i:i+BATCH],
        template='(%s,%s,%s,%s,%s,%s,%s,%s,%s::vector,%s)',
    )
    conn.commit()
    print(f'  Loaded {min(i+BATCH, len(rows)):>6} / {len(rows)}')

# ── Build indexes AFTER loading ───────────────────────────────────────────────
# lists = sqrt(N) rounded; for ~14k rows → 50 is appropriate
print('Building IVFFlat vector index (lists=50)...')
cur.execute('''
    CREATE INDEX ON chunks USING ivfflat (embedding vector_cosine_ops)
    WITH (lists = 50);
''')
print('Building GIN full-text index...')
cur.execute('CREATE INDEX ON chunks USING GIN (content_tsv);')
conn.commit()

cur.execute('SELECT COUNT(*) FROM chunks;')
print(f'\nTotal rows in DB: {cur.fetchone()[0]}')
print('Done!')

## Step 8 — Hybrid retrieval + 10 demo queries

Score formula (from proposal):  
```
score = α × cosine_similarity(q, chunk) + (1 − α) × ts_rank(chunk, query)
```
Default α = 0.7

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

def hybrid_search(
    query:        str,
    alpha:        float = ALPHA,
    top_k:        int   = 5,
    sector:       str   = None,
    filing_type:  str   = None,
    ticker:       str   = None,
    fiscal_year:  int   = None,
):
    """
    Hybrid retrieval: alpha*dense_cosine + (1-alpha)*BM25_ts_rank.
    Optional WHERE filters: sector, filing_type, ticker, fiscal_year.
    """
    q_emb = model.encode([query], normalize_embeddings=True)[0].tolist()

    filters, params = [], []
    if sector:      filters.append('sector = %s');      params.append(sector)
    if filing_type: filters.append('filing_type = %s'); params.append(filing_type)
    if ticker:      filters.append('ticker = %s');      params.append(ticker)
    if fiscal_year: filters.append('fiscal_year = %s'); params.append(fiscal_year)

    where_extra = ('AND ' + ' AND '.join(filters)) if filters else ''

    sql = f'''
        SELECT
            ticker, sector, filing_type, fiscal_year,
            content, source_url,
            (
                {alpha}   * (1 - (embedding <=> %s::vector))
              + {1-alpha:.2f} * ts_rank(content_tsv,
                                    plainto_tsquery('english', %s))
            ) AS score
        FROM chunks
        WHERE content_tsv @@ plainto_tsquery('english', %s)
        {where_extra}
        ORDER BY score DESC
        LIMIT {top_k};
    '''

    all_params = [q_emb, query, query] + params
    cur.execute(sql, all_params)
    return cur.fetchall()


def print_results(query, results, show_content_chars=200):
    print(f'\n{"="*72}')
    print(f'Query : {query}')
    print(f'{"─"*72}')
    if not results:
        print('  (no results)')
        return
    for ticker, sector, ftype, fy, content, url, score in results:
        print(f'  [{score:.4f}] {ticker} | {sector} | {ftype} | FY{fy}')
        print(f'  {content[:show_content_chars].strip()}...')
        print(f'  {url}')
        print()

print('hybrid_search() ready.')

In [ ]:
# ── 10 demo queries stratified across sectors + filing types ─────────────────
DEMO_QUERIES = [
    # Banking (2)
    ('What are JPMorgan Chase main credit risk factors?',
     dict(sector='banking', filing_type='10-K')),
    ('How did Goldman Sachs manage provision for credit losses?',
     dict(sector='banking', filing_type='10-Q')),

    # Technology (2)
    ('What drove Microsoft Azure cloud segment revenue growth?',
     dict(ticker='MSFT')),
    ('What is NVIDIA revenue breakdown from data center versus gaming?',
     dict(ticker='NVDA')),

    # Healthcare (2)
    ('Describe Pfizer R&D pipeline spending and clinical trial risks.',
     dict(sector='healthcare')),
    ('How does UnitedHealth Group describe its medical cost ratio?',
     dict(ticker='UNH', filing_type='10-K')),

    # Energy (2)
    ('How did ExxonMobil describe its capital expenditure and upstream plan?',
     dict(sector='energy', filing_type='10-K')),
    ('How did energy companies respond to crude oil price volatility in 2022?',
     dict(sector='energy', fiscal_year=2022)),

    # Consumer (2)
    ('What is Amazon trend in operating income across business segments?',
     dict(ticker='AMZN')),
    ('How does Walmart describe supply chain risk and inventory management?',
     dict(ticker='WMT')),
]

for i, (query, filters) in enumerate(DEMO_QUERIES, 1):
    print(f'\n>>> Demo query {i:02d} | filters: {filters}')
    results = hybrid_search(query, top_k=3, **filters)
    print_results(query, results)

## Bonus — Corpus statistics for writeup

In [ ]:
cur.execute('''
    SELECT
        sector,
        filing_type,
        COUNT(*)               AS chunks,
        COUNT(DISTINCT ticker) AS companies
    FROM chunks
    GROUP BY sector, filing_type
    ORDER BY sector, filing_type;
''')

print(f'{"Sector":<15} {"Type":<8} {"Chunks":>8} {"Companies":>10}')
print('-' * 45)
for sector, ftype, n_chunks, n_co in cur.fetchall():
    print(f'{sector:<15} {ftype:<8} {n_chunks:>8} {n_co:>10}')

cur.execute('SELECT COUNT(*), pg_size_pretty(pg_total_relation_size(oid)) FROM pg_class WHERE relname=%s', ('chunks',))
n, sz = cur.fetchone()
print(f'\nTotal chunks: {n}  |  Table size: {sz}')

## Optional — Export pg_dump to Drive (for persistence)

In [ ]:
dump_path = f'{BASE_DIR}/rag_db.dump'
!PGPASSWORD=rag_pass pg_dump -U rag_user -h localhost -Fc rag_db -f {dump_path}
print(f'Dump saved → {dump_path}')

# To restore in a new session:
# !PGPASSWORD=rag_pass pg_restore -U rag_user -h localhost -d rag_db {dump_path}